# PART B: External data, wikipedia page views api

## Time spent: 30 mins

### Notes:
- This is the beginning of developing a script to get wikipedia page views, I wrote 2 functions:
    - `get_correct_wikipedia_title`: check for correct wikipedia page name
    - `get_wikipedia_pageviews`: get daily page views for that page across 2023 (period of the releases dataset)
- Used the documentation here for finding correct named page: https://www.mediawiki.org/wiki/API:Search
- Used the documentation here for pageviews: https://doc.wikimedia.org/generated-data-platform/aqs/analytics-api/reference/page-views.html
- Saved the outputs in a pickled json file saved locally from which you can generate the features you needs/conduct extra analysis (I haven't done this because of time limit but those would be my next steps)

In [69]:
import pandas as pd
import numpy as np
import requests
import pickle
import time
df = pd.read_csv("releases.csv")

In [ ]:
def get_correct_wikipedia_title(title):
    search_url = "https://en.wikipedia.org/w/api.php"
    
    params = {
        "action": "query",
        "list": "search",
        "srsearch": f"{title} film 2023",
        "format": "json"
    }

    headers = {"User-Agent": "movie-analytics-project"}

    r = requests.get(search_url, params=params, headers=headers)
    data = r.json()
    
    if len(data["query"]["search"]) == 0:
        params = {
            "action": "query",
            "list": "search",
            "srsearch": f"{title}", # adjusted srsearch just in case first one misses
            "format": "json"
        }

        headers = {"User-Agent": "movie-analytics-project"}

        r = requests.get(search_url, params=params, headers=headers)
        data = r.json()

        if len(data["query"]["search"]) == 0: 
            return None
    
    return data["query"]["search"][0]["title"]

In [76]:
def get_wikipedia_pageviews(title, start_date, end_date):
    title = title.replace(" ", "_")
    
    url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia.org/all-access/user/{title}/daily/{start_date}/{end_date}"
    
    headers = {"User-Agent": "movie-analytics-project"}
    
    r = requests.get(url, headers=headers)
    
    if r.status_code != 200:
        return None
    
    data = r.json()["items"]
    
    df = pd.DataFrame(data)
    df["views"] = df["views"].astype(int)
    
    return df

In [78]:
wiki_page_views = {} 
count = 0
for i in df["title"].values:
    try: 
        correct_page_name = get_correct_wikipedia_title(i)
        page_views = get_wikipedia_pageviews(correct_page_name, "2023010100", "2024010100") # date range of df
    except:
        page_views = None
        print(f"Error processing {i}.")
    wiki_page_views[i] = page_views
    count +=1
    if count % 20 == 0:
        print(f'Films processed: {count}/{len(df["title"].values)}')
    time.sleep(1) # tried without sleep and got some errors, 1 second break got rid of all of them


Films processed: 20/273
Films processed: 40/273
Films processed: 60/273
Films processed: 80/273
Films processed: 100/273
Films processed: 120/273
Films processed: 140/273
Films processed: 160/273
Films processed: 180/273
Films processed: 200/273
Films processed: 220/273
Films processed: 240/273
Films processed: 260/273


In [79]:
with open("wiki_daily_page_views.pkl", "wb") as f:
    pickle.dump(wiki_page_views, f)

In [92]:
with open("wiki_daily_page_views.pkl", "rb") as f:
    wiki_pickle = pickle.load(f)
wiki_pickle_df = pd.DataFrame({"title":wiki_pickle.keys(), "timeseries_df":wiki_pickle.values()})

In [93]:
len(wiki_pickle)

273

In [94]:
wiki_pickle["Blue Beetle"]

,project,article,granularity,timestamp,access,agent,views
0,en.wikipedia,Blue_Beetle_(film),daily,2023010100,all-access,user,2679
1,en.wikipedia,Blue_Beetle_(film),daily,2023010200,all-access,user,3040
2,en.wikipedia,Blue_Beetle_(film),daily,2023010300,all-access,user,3050
3,en.wikipedia,Blue_Beetle_(film),daily,2023010400,all-access,user,3259
4,en.wikipedia,Blue_Beetle_(film),daily,2023010500,all-access,user,2714
...,...,...,...,...,...,...,...
361,en.wikipedia,Blue_Beetle_(film),daily,2023122800,all-access,user,9525
362,en.wikipedia,Blue_Beetle_(film),daily,2023122900,all-access,user,8891
363,en.wikipedia,Blue_Beetle_(film),daily,2023123000,all-access,user,10158
364,en.wikipedia,Blue_Beetle_(film),daily,2023123100,all-access,user,10485
